### Supplement 2: Routing (`2-routing.py`)

Send each request to the right handler: One LLM call sorts the request into a type, and your code calls the function that deals with that type.

**The key point:** The router is an LLM call that returns a label, `new_event`, `modify_event` or `other`. A plain `if`/`elif` in your code reads the label and calls one handler. As in prompt chaining, the model never decides what runs next; it only fills in the label your code branches on. The difference is that chaining runs every step, while routing runs only one branch ([Supplement_1-prompt-chaining.ipynb](Supplement_1-prompt-chaining.ipynb)).

No real calendar events are changed. The handlers only return a message saying what would have been done.

<pre style="font-size: 0.85em; line-height: 1.35; white-space: pre; overflow-x: auto;">
user_input
   │
   ▼
Router   route_calendar_request   → CalendarRequestType
   │     request_type, confidence_score, description
   ▼
Gate     Confidence below 0.7 → stop, return None
   │
   ├── "new_event"     → handle_new_event      → CalendarResponse
   ├── "modify_event"  → handle_modify_event   → CalendarResponse
   └── "other"         → return None
</pre>

#### 1. Setup
Same as Supplement 1: `load_dotenv()` puts `OPENAI_API_KEY` into `os.environ`, and `OpenAI()` reads it from there.

In [1]:
from typing import Optional, Literal
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
model = "gpt-5-nano"

#### 2. The data models
- `CalendarRequestType`: The router's answer. `Literal["new_event", "modify_event", "other"]` becomes a list of allowed values (`enum`) in the JSON Schema, so the model has to pick one of those three strings.
- `NewEventDetails` and `ModifyEventDetails`: What each handler asks the model to pull out. `ModifyEventDetails` holds a list of `Change` objects, one per field to change.
- `CalendarResponse`: What every handler returns. Your code builds it, not the model.

In [2]:
# Define the data models for routing and responses

class CalendarRequestType(BaseModel):
    """Router LLM call: Determine the type of calendar request"""

    request_type: Literal["new_event", "modify_event", "other"] = Field(
        description="Type of calendar request being made"
    )
    confidence_score: float = Field(description="Confidence score between 0 and 1")
    description: str = Field(description="Cleaned description of the request")


class NewEventDetails(BaseModel):
    """Details for creating a new event"""

    name: str = Field(description="Name of the event")
    date: str = Field(description="Date and time of the event (ISO 8601)")
    duration_minutes: int = Field(description="Duration in minutes")
    participants: list[str] = Field(description="List of participants")


class Change(BaseModel):
    """Details for changing an existing event"""

    field: str = Field(description="Field to change")
    new_value: str = Field(description="New value for the field")


class ModifyEventDetails(BaseModel):
    """Details for modifying an existing event"""

    event_identifier: str = Field(
        description="Description to identify the existing event"
    )
    changes: list[Change] = Field(description="List of changes to make")
    participants_to_add: list[str] = Field(description="New participants to add")
    participants_to_remove: list[str] = Field(description="Participants to remove")


class CalendarResponse(BaseModel):
    """Final response format"""

    success: bool = Field(description="Whether the operation was successful")
    message: str = Field(description="User-friendly response message")
    calendar_link: Optional[str] = Field(description="Calendar link if applicable")

#### 3. The router
`route_calendar_request` asks the model to sort the request into one of the three types and fill a `CalendarRequestType`. The cells after it run the router once on the new-event sentence.

In [3]:
# Define the routing and processing functions

def route_calendar_request(user_input: str) -> CalendarRequestType:
    """Router LLM call to determine the type of calendar request"""
    print("Routing calendar request")

    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Determine if this is a request to create a new calendar event or modify an existing one.",
            },
            {"role": "user", "content": user_input},
        ],
        response_format=CalendarRequestType,
    )
    result = completion.choices[0].message.parsed
    print(
        f"Request routed as: {result.request_type} with confidence: {result.confidence_score}"
    )
    return result

In [4]:
new_event_input = "Let's schedule a team meeting next Tuesday at 2pm with Alice and Bob"

route = route_calendar_request(new_event_input)

Routing calendar request
Request routed as: new_event with confidence: 0.88


In [5]:
print(type(route))
print(route.model_dump_json(indent=2))

<class '__main__.CalendarRequestType'>
{
  "request_type": "new_event",
  "confidence_score": 0.88,
  "description": "Create a new calendar event: Team meeting next Tuesday at 2:00 PM with Alice and Bob."
}


#### 4. The new-event handler
`handle_new_event` receives the router's `description`, not the original sentence. It makes one more LLM call to get `NewEventDetails`, then builds the `CalendarResponse` itself: Only the details come from the model, while `success=True` and the `calendar://` link are written by your code.

In [6]:
def handle_new_event(description: str) -> CalendarResponse:
    """Process a new event request"""
    print("Processing new event request")

    # Get event details
    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Extract details for creating a new calendar event.",
            },
            {"role": "user", "content": description},
        ],
        response_format=NewEventDetails,
    )
    details = completion.choices[0].message.parsed

    print(f"New event: {details.model_dump_json(indent=2)}")

    # Generate response
    return CalendarResponse(
        success=True,
        message=f"Created new event '{details.name}' for {details.date} with {', '.join(details.participants)}",
        calendar_link=f"calendar://new?event={details.name}",
    )

In [7]:
new_event_response = handle_new_event(route.description)

Processing new event request
New event: {
  "name": "Team meeting",
  "date": "2026-10-27T14:00:00",
  "duration_minutes": 60,
  "participants": [
    "Alice",
    "Bob"
  ]
}


In [8]:
print(type(new_event_response))
print(new_event_response.model_dump_json(indent=2))

<class '__main__.CalendarResponse'>
{
  "success": true,
  "message": "Created new event 'Team meeting' for 2026-10-27T14:00:00 with Alice, Bob",
  "calendar_link": "calendar://new?event=Team meeting"
}


#### 5. The modify-event handler
The same shape as `handle_new_event`, with `ModifyEventDetails` instead. Section 8 runs it.

In [9]:
def handle_modify_event(description: str) -> CalendarResponse:
    """Process an event modification request"""
    print("Processing event modification request")

    # Get modification details
    completion = client.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "Extract details for modifying an existing calendar event.",
            },
            {"role": "user", "content": description},
        ],
        response_format=ModifyEventDetails,
    )
    details = completion.choices[0].message.parsed

    print(f"Modified event: {details.model_dump_json(indent=2)}")

    # Generate response
    return CalendarResponse(
        success=True,
        message=f"Modified event '{details.event_identifier}' with the requested changes",
        calendar_link=f"calendar://modify?event={details.event_identifier}",
    )

#### 6. The routing, in plain code
`process_calendar_request` runs the router and stops if `confidence_score` is below 0.7. Then an `if`/`elif` on `request_type` calls one handler, and `"other"` falls through to `else` and returns `None`.

In [10]:
def process_calendar_request(user_input: str) -> Optional[CalendarResponse]:
    """Main function implementing the routing workflow"""
    print("Processing calendar request")

    # Route the request
    route_result = route_calendar_request(user_input)

    # Check confidence threshold
    if route_result.confidence_score < 0.7:
        print(f"Low confidence score: {route_result.confidence_score}")
        return None

    # Route to appropriate handler
    if route_result.request_type == "new_event":
        return handle_new_event(route_result.description)
    elif route_result.request_type == "modify_event":
        return handle_modify_event(route_result.description)
    else:
        print("Request type not supported")
        return None

#### 7. A new event: The router picks `handle_new_event`

In [11]:
# Test with new event

new_event_input = "Let's schedule a team meeting next Tuesday at 2pm with Alice and Bob"
result = process_calendar_request(new_event_input)
if result:
    print(f"Response: {result.message}")

Processing calendar request
Routing calendar request
Request routed as: new_event with confidence: 0.92
Processing new event request
New event: {
  "name": "Team meeting",
  "date": "2026-10-27T14:00:00",
  "duration_minutes": 60,
  "participants": [
    "Alice",
    "Bob"
  ]
}
Response: Created new event 'Team meeting' for 2026-10-27T14:00:00 with Alice, Bob


#### 8. A change to an existing event: The router picks `handle_modify_event`

In [12]:
# Test with modify event

modify_event_input = (
    "Can you move the team meeting with Alice and Bob to Wednesday at 3pm instead?"
)
result = process_calendar_request(modify_event_input)
if result:
    print(f"Response: {result.model_dump()}")

Processing calendar request
Routing calendar request
Request routed as: modify_event with confidence: 0.84
Processing event modification request
Modified event: {
  "event_identifier": "Team meeting with Alice and Bob",
  "changes": [
    {
      "field": "day_of_week",
      "new_value": "Wednesday"
    },
    {
      "field": "time",
      "new_value": "3:00 PM"
    }
  ],
  "participants_to_add": [],
  "participants_to_remove": []
}
Response: {'success': True, 'message': "Modified event 'Team meeting with Alice and Bob' with the requested changes", 'calendar_link': 'calendar://modify?event=Team meeting with Alice and Bob'}


#### 9. Not a calendar request: No handler runs
The router labels it `other`, so `process_calendar_request` returns `None` after a single API call.

In [13]:
# Test with invalid request

invalid_input = "What's the weather like today?"
result = process_calendar_request(invalid_input)
if not result:
    print("Request not recognized as a calendar operation")

Processing calendar request
Routing calendar request
Request routed as: other with confidence: 0.92
Request type not supported
Request not recognized as a calendar operation
